# Input files

AmpSeeker requires only: 
- Sample metadata file `.tsv` or SampleSheet.csv in Illumina run folder containing metadata
- Bed file of amplicon target SNP coordinates `.bed`
- Paired-end Illumina Amplicon-Seq data `.fastq.gz` OR single-end Nanopore amplicon data `.fastq.gz`
- Reference genome files

**Sample metadata file**

If working directly from the Illumina BCL folder, a SampleSheet (`SampleSheet.csv`) must be placed in the Illumina run folder. An example sample sheet is located at `resources/exampleSampleSheet.csv`.

If working directly from fastq files, the user must provide either a SampleSheet or a sample metadata file in `.tsv` format which should be placed in the `config/` folder and pointed to in the user's `config.yaml`. The default name is `metadata.tsv`.

**File format:** the metadata file **must be tab-separated** (`.tsv`) — columns are separated by a single tab character (`\t`), not commas or spaces. Do not use Excel's default `.csv` export. If editing in Excel or a similar spreadsheet tool, save explicitly as "Tab-separated values (.tsv)". The first row must be a header row containing the column names below.

### Required column

Only one column is strictly required:

| Column      | Description |
|-------------|-------------|
| `sample_id` | Unique identifier for each sample. Must match the fastq filenames if using `fastq.auto: True` (i.e. `{sample_id}_1.fastq.gz` / `{sample_id}_2.fastq.gz` for Illumina, or `{sample_id}.fastq.gz` for Nanopore). No spaces or special characters. |

In practice you will almost always also want **at least one grouping column** (e.g. `location`, `taxon`, `country`) so that samples can be split and coloured in downstream plots. In AmpSeeker, a cohort simply means the set of samples sharing the same value in one of these grouping columns.

### Cohort column naming rules

Any metadata column can be used as a cohort column **provided that**:
- its name is listed under `cohort-columns` in `config.yaml`
- the name matches the metadata header **exactly**, including capitalization
- it is a simple column name beginning with a letter and containing only letters, numbers, and underscores

For example, `location`, `taxon`, `country`, and `collection_year` are safe choices. Avoid spaces, hyphens, punctuation, or column names beginning with a number.

### Optional columns

Any additional columns can be added freely. The following are recognised by specific parts of the workflow:

| Column                         | Used by                                 | Purpose |
|--------------------------------|-----------------------------------------|---------|
| `location`, `taxon`, `country` (or any categorical column) | Population structure, allele frequencies, genetic diversity, PCA plots | Any column listed under `cohort-columns` in `config.yaml` is used to define cohorts, meaning groups of samples with the same value in that column. These columns are used to colour points in PCA and coverage plots, to compute per-cohort allele frequencies, and to split genetic diversity statistics. |
| `latitude`, `longitude`        | `analysis.sample-map`                   | Required *only* if `analysis.sample-map: True` in `config.yaml`. Used to plot sample collection locations on an interactive map. Must be decimal degrees (e.g. `5.668`, `-0.167`). Omit these columns if you do not want a sample map. |
| `fq1`, `fq2`                   | Illumina input (when `fastq.auto: False`) | Paths to the forward (`fq1`) and reverse (`fq2`) fastq files for each sample. Only needed if `fastq.auto: False` in `config.yaml`; otherwise AmpSeeker auto-discovers fastqs from `resources/reads/`. |
| `fq1`                          | Nanopore input (when `fastq.auto: False`) | Path to the single-end fastq file for each sample. |
| Any other column (e.g. `year`, `strain`, `plate`, `well`, `insecticide`) | Results notebooks / reports | Passed through to the results notebooks and tables. Safe to include — unknown columns are ignored by the pipeline but preserved in the output for your own reference. |

### Example `metadata.tsv` (Illumina, paired-end)

This matches the example metadata shipped in the repository at `tests/snakemake/config/metadata.tsv`:

| sample_id  | country | location     | taxon    | latitude | longitude | fq1                                    | fq2                                    |
|------------|---------|--------------|----------|----------|-----------|----------------------------------------|----------------------------------------|
| ERR3058522 | Ghana   | Madina_North | gambiae  | 5.668    | -0.167    | resources/reads/ERR3058522_1.fastq.gz  | resources/reads/ERR3058522_2.fastq.gz  |
| ERR3058532 | Ghana   | Madina_North | gambiae  | 5.668    | -0.167    | resources/reads/ERR3058532_1.fastq.gz  | resources/reads/ERR3058532_2.fastq.gz  |
| ERR3058982 | Benin   | Avrankou     | coluzzii | 6.45     | 2.25      | resources/reads/ERR3058982_1.fastq.gz  | resources/reads/ERR3058982_2.fastq.gz  |

In this example only `sample_id` is strictly required; `country`, `location` and `taxon` are optional grouping columns used for cohort analyses, `latitude`/`longitude` are only needed if drawing a sample map, and `fq1`/`fq2` are only needed when `fastq.auto: False`.

### Minimal `metadata.tsv`

If you have placed fastqs in `resources/reads/` with auto-naming (`fastq.auto: True`) and do not need a sample map, the file can be as small as:

| sample_id  | location     |
|------------|--------------|
| ERR3058522 | Madina_North |
| ERR3058982 | Avrankou     |

--- 

**Bed file of amplicon targets**

Users should also provide a file in bed format, with at least 5 columns (chromosome, start, end, amplicon_id, target_label), and optionally reference and alternative alleles. The file should have **no header** and must be **tab-separated**. The bed file contains the coordinates of the amplicon targets and should be placed in the `config/` folder and pointed to in the user's `config.yaml`. The default name is `amplicon_targets.bed`. 

| 2L      | 209535  | 209536  | Agam_1  | AIM1  | A  | G |
|---------|---------|---------|---------|-------|----|----|
| 2L      | 927246  | 927247  | Agam_2  | AIM2  | C  | A |
| 2L      | 1274352 | 1274353 | Agam_3  | AIM3  | G  | A |
| 2L      | 1418209 | 1418210 | Agam_4  | AIM4  | T  | C |
| 2L      | 1571928 | 1571929 | Agam_5  | AIM5  | T  | C |

The columns are:
1. **Chromosome**: Genomic chromosome or contig name (e.g., "2L")
2. **Start position**: 0-based start coordinate of the target
3. **End position**: End coordinate of the target (typically start+1 for SNPs)
4. **Amplicon ID**: Identifier for the amplicon (e.g., "Agam_1")
5. **Target label**: Descriptive name for the target (e.g., "AIM1")
6. **Reference allele**: (Optional) The reference allele at this position
7. **Alternate allele(s)**: (Optional) The alternate allele(s) at this position

The `target_label` column values should be unique for each row of the bed file. The `amplicon_id` column values can be repeated for multiple rows, to indicate that those SNPs are located on the same amplicon.

---

## Sequencing Data Input

### Illumina Data

**Illumina run folders or Paired-end Amplicon-Sequencing fastq reads**

Users can either provide the path to an Illumina MiSeq run folder, or provide paired-end fastq reads. If converting BCL files to fastq within the workflow, a `SampleSheet.csv` must be placed in the Illumina run folder. The SampleSheet must have the CreateFastqForIndexReads parameter set to 1, please see the `exampleSampleSheet.csv` in the `resources/` directory.

#### SampleSheet.csv Structure

The SampleSheet.csv is a structured file with several sections:
- **[Header]**: Contains experiment metadata
- **[Reads]**: Defines read lengths (typically 151 for paired-end reads)
- **[Settings]**: Contains critical parameters including `CreateFastqForIndexReads=1`
- **[Data]**: Sample information including IDs, indexes, and metadata

The `CreateFastqForIndexReads=1` setting in the [Settings] section is essential as it instructs the BCL conversion to generate FASTQ files for index reads, which are used for demultiplexing samples.

Example [Data] section format (comma-separated, as per the Illumina SampleSheet specification):

| sample_ID | sample_name | index    | index2   | well | plate_name | taxon | location | country | latitude | longitude |
|-----------|-------------|----------|----------|------|------------|-------|----------|---------|----------|-----------|
| GH_01     | GH_01       | ATCACGTT | CCTATCCT | A1   | 3          |       | Obuasi   | Ghana   |          |           |
| GH_02     | GH_02       | CGATGTTT | CCTATCCT | A2   | 3          |       | Obuasi   | Ghana   |          |           |

Required columns for the [Data] section:
- **sample_ID**: Unique sample identifier, used to name output files
- **sample_name**: Name displayed in reports (often the same as sample_ID)
- **index**: Forward index sequence for demultiplexing
- **index2**: Reverse index sequence for demultiplexing (for dual indexing)

Optional metadata columns can be added (e.g., well, plate_name, taxon, location, country, latitude, longitude) and are treated exactly like the optional columns in `metadata.tsv` described above — they will be incorporated into analysis results and can be referenced via `cohort-columns` in `config.yaml`.

#### FASTQ File Specifications for Illumina

If providing fastq files, two gzipped fastq files for each sample are required, one for each pair of paired-end reads. Reads can be already trimmed or AmpSeeker can trim them, using the fastp module (configurable in the config.yaml).

Two options are available for specifying the location of the fastq files. Either the metadata contains two columns `fq1` and `fq2` with the paths to the fastq files, or the fastq files are placed in the following directory (`resources/reads/`) with the following naming pattern:

``` 
ampseeker_dir/resources/reads/

Reads should be named as `{sample_id}_1.fastq.gz`, `{sample_id}_2.fastq.gz`.
```

If providing fastq paths in the metadata file, they can be named anything. 

### Nanopore Data
#### FASTQ File Specifications for Nanopore

For Nanopore data, provide single gzipped fastq files for each sample. The metadata file should contain a single column `fq1` with paths to the fastq files, or place files in `resources/reads/` with the naming pattern `{sample_id}.fastq.gz`.

**Nanopore metadata.tsv format** (tab-separated):

| sample_id | fq1                    | location | taxon    |
|-----------|------------------------|----------|----------|
| ContTia1  | reads/ContTia1.fq.gz   | Tiassale | coluzzii |
| ContTia2  | reads/ContTia2.fq.gz   | Tiassale | coluzzii |

Or with automatic file detection:
```
ampseeker_dir/resources/reads/
ContTia1.fastq.gz
ContTia2.fastq.gz
```

---

**Reference genome files**

For both platforms, genome alignment requires a fasta file containing the genome sequence. All input `.fa` files can be gzipped `.fa.gz`.   

Reference genomes can be downloaded from vectorbase with the `resources/reference/download-vectorbase-reference.sh` script. Run it from the root AmpSeeker directory.

The user provides the path to the reference files in the configuration file (`config.yaml`).

1. **Genome chromosomes reference file (.fa/.fa.gz)**. Contains the DNA sequence for the genome in fasta format.
2. **Genome feature file (.gff3 format)**. 

Ensure that contigs in the reference genome match the contigs in the bed file. 

*Note - Genome reference files from VectorBase now have prefixes before each contig name, such as 'AgamP4_2L'. Either the bed file can be updated to match these, or the names in the reference files can be modified. The downloader script can strip these prefixes automatically.